# Promotion uplift: which promotions actually create extra sales?

This notebook reads the results of `sql/05_uplift.sql` and `sql/06_cannibalisation.sql` from PostgreSQL, charts them, and runs a placebo test to check that my method doesn't invent uplift.

All dollar amounts come from the 2,500 loyalty households in the data, not whole-store sales, so they look small. What matters here is the comparison between promotion types.

In [ ]:
import os
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, URL

# asks for my PostgreSQL password, so it's never saved in the notebook
password = os.environ.get("PG_PASSWORD") or getpass("PostgreSQL password: ")
engine = create_engine(URL.create("postgresql+psycopg2", username="postgres", password=password,
                                  host="localhost", port=5432, database="retail_promo"))

uplift = pd.read_sql("SELECT * FROM analysis.uplift_summary", engine)
cannib = pd.read_sql("SELECT * FROM analysis.cannibalisation_summary", engine)
uplift.head()

In [ ]:
# chart style: one quiet look for every chart in the project
BLUE, GREY, INK, MUTED, SURFACE = "#2a78d6", "#a3a29c", "#0b0b0b", "#52514e", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#d6d5d0", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": "#e9e8e4", "grid.linewidth": 0.8,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
os.makedirs("../docs/charts", exist_ok=True)

DISPLAY = {"1": "Store front", "2": "Store rear", "3": "Front end cap", "4": "Mid-aisle end cap",
           "5": "Rear end cap", "6": "Side-aisle end cap", "7": "In-aisle", "9": "Secondary location",
           "A": "In-shelf"}
MAILER = {"A": "Interior page feature", "C": "Interior page line item", "D": "Front page feature",
          "F": "Back page feature", "H": "Wrap front feature", "J": "Wrap interior coupon",
          "L": "Wrap back feature"}
# X, Z and P are "free item" placements (or almost never used). A free item has $0 sales,
# so measuring it in dollars makes no sense - I leave them out of the ranking.

def bar_chart(df, label_col, value_col, ci_col, title, xlabel, highlight=None, file=None):
    """Horizontal bars with a 95% range, biggest at the top."""
    df = df.sort_values(value_col)
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(df) + 1.2))
    colors = [BLUE if (highlight is None or lbl in highlight) else GREY for lbl in df[label_col]]
    ax.barh(df[label_col], df[value_col], color=colors, height=0.6)
    if ci_col:
        ax.errorbar(df[value_col], df[label_col], xerr=df[ci_col], fmt="none", ecolor=INK, elinewidth=1, capsize=3)
    # value label just past the end of the bar or its range line, so they never overlap
    ends = df[ci_col] if ci_col else pd.Series(0, index=df.index)
    for y, v, e in zip(df[label_col], df[value_col], ends):
        x, ha = (v + e, "left") if v >= 0 else (v - e, "right")
        ax.text(x, y, f"  {v:,.2f}  ", va="center", ha=ha, color=INK, fontsize=9)
    ax.margins(x=0.15)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.tick_params(axis="y", length=0)
    ax.axvline(0, color="#d6d5d0", linewidth=1)
    fig.tight_layout()
    if file:
        fig.savefig(f"../docs/charts/{file}", dpi=150)
    plt.show()

## 1. The mailer creates most of the extra sales, and display adds on top

`avg_extra_per_week` is how much more a product sells in one store in a promo week than in a normal week. The black lines are the 95% range.

In [ ]:
by_type = uplift[uplift["level"] == "1. promo type"].copy()
by_type["share_of_extra"] = by_type["extra_sales"] / by_type["extra_sales"].sum()
by_type[["placement", "promo_weeks", "extra_sales", "share_of_extra", "uplift_pct",
         "avg_extra_per_week", "ci95", "extra_per_discount_dollar"]]

In [ ]:
bar_chart(by_type, "placement", "avg_extra_per_week", "ci95",
          "Extra sales per promo week, by promotion type",
          "Extra sales per product, per store, per week ($)", file="uplift_by_type.png")

t = by_type.set_index("placement")
mailer, display, both = (t.loc[k, "avg_extra_per_week"] for k in ["mailer only", "display only", "display + mailer"])
print(f"Mailer only creates {t.loc['mailer only', 'share_of_extra']:.0%} of all extra sales.")
print(f"Display + mailer (${both:.2f}) is almost exactly mailer (${mailer:.2f}) + display (${display:.2f}) = ${mailer + display:.2f}:"
      " the two effects simply add up.")

## 2. Where the display sits barely matters. Where the mailer puts it matters a lot

In [ ]:
disp = uplift[uplift["level"].str.startswith("2.")].copy()
disp["location"] = disp["placement"].map(DISPLAY)
bar_chart(disp, "location", "avg_extra_per_week", "ci95",
          "Display location: extra sales per promo week",
          "Extra sales per product, per store, per week ($)", file="uplift_by_display.png")
print(f"All {len(disp)} display locations fall between ${disp['avg_extra_per_week'].min():.2f}"
      f" and ${disp['avg_extra_per_week'].max():.2f} per week, and their ranges overlap.")

In [ ]:
mail = uplift[uplift["level"].str.startswith("3.") & uplift["placement"].isin(MAILER.keys())].copy()
mail["position"] = mail["placement"].map(MAILER)
bar_chart(mail, "position", "avg_extra_per_week", "ci95",
          "Mailer placement: extra sales per promo week",
          "Extra sales per product, per store, per week ($)",
          highlight={"Front page feature", "Interior page feature"}, file="uplift_by_mailer.png")

m = mail.set_index("placement")
share_interior = m.loc["A", "promo_weeks"] / mail["promo_weeks"].sum()
print(f"Front page feature: ${m.loc['D', 'avg_extra_per_week']:.2f} per week, "
      f"{m.loc['D', 'avg_extra_per_week'] / m.loc['A', 'avg_extra_per_week']:.1f}x the interior page feature "
      f"(${m.loc['A', 'avg_extra_per_week']:.2f}), which takes {share_interior:.0%} of mailer-only weeks.")

## 3. Per dollar of discount, displays are the cheapest extra sales

`extra_per_discount_dollar` = extra sales ÷ discount given in those promo weeks. It counts all discount in those weeks, including discount that would have happened anyway, so it's a rough measure. And it's sales, not profit.

In [ ]:
eff = pd.concat([
    by_type.assign(name=by_type["placement"]),
    disp.assign(name="Display: " + disp["location"]),
    mail.assign(name="Mailer: " + mail["position"]),
])
bar_chart(eff, "name", "extra_per_discount_dollar", None,
          "Extra sales per $1 of discount",
          "Extra sales ($) per $1 of discount given", highlight=set(by_type["placement"]),
          file="extra_per_discount_dollar.png")

## 4. Cannibalisation: how much of the extra sales was taken from similar products?

`taken_from_similar` = extra sales of the promoted products minus extra sales of the whole sub-category in that store and week. Positive means the promoted item took sales from its neighbours. Negative means the whole sub-category grew by more than the promoted item (a "halo").

In [ ]:
c = cannib[cannib["level"].isin(["1. overall", "2. promo type"])].copy()
c["cannibalised_share"] = c["taken_from_similar"] / c["promoted_extra"]
c["share_ci95"] = c["taken_ci95"] / c["promoted_extra"].abs()
c["clear_effect"] = (c["taken_from_similar"].abs() > c["taken_ci95"])
c[["grp", "promoted_extra", "category_extra", "taken_from_similar", "taken_ci95",
   "cannibalised_share", "clear_effect", "coverage_pct"]]

In [ ]:
bar_chart(c[c["level"] == "2. promo type"], "grp", "cannibalised_share", "share_ci95",
          "Share of extra sales taken from similar products",
          "Share of the promoted products' extra sales (negative = whole sub-category grew more)",
          file="cannibalisation_by_type.png")

In [ ]:
dept = cannib[(cannib["level"] == "3. department") & (cannib["subcat_stores"] >= 500)].copy()
dept["cannibalised_pct_clear"] = np.where(dept["taken_from_similar"].abs() > dept["taken_ci95"],
                                          dept["cannibalised_pct"], np.nan)
dept.sort_values("promoted_extra", ascending=False)[
    ["grp", "subcat_stores", "promoted_extra", "taken_from_similar", "taken_ci95",
     "cannibalised_pct", "cannibalised_pct_clear"]]

`cannibalised_pct_clear` only shows a number when the range doesn't include zero. Blank means I can't tell from this data.

## 5. Placebo test: does my method invent uplift?

If I pretend some **normal** weeks were promo weeks and run the same calculation, the "uplift" should be about zero. If it isn't, my method is biased.

For every product-store in the uplift table, I take its normal (non-promoted) weeks, randomly label up to 10 of them as fake promo weeks, build the baseline from the rest, and measure the fake uplift. I repeat this 20 times with different random picks.

In [ ]:
# weekly sales in normal weeks only, plus the list of real promo weeks for each product-store
normal_sales = pd.read_sql("""
    SELECT s.product_id, s.store_id, s.week_no, s.adj_sales
    FROM analysis.ps_weeks s
    JOIN analysis.baseline b ON b.product_id = s.product_id AND b.store_id = s.store_id
    WHERE NOT s.promoted
""", engine)
promo = pd.read_sql("SELECT DISTINCT product_id, store_id, week_no FROM analysis.promo_week_uplift", engine)
pairs = pd.read_sql("SELECT product_id, store_id FROM analysis.baseline", engine)
print(f"{len(pairs):,} product-stores, {len(normal_sales):,} normal weeks with a sale")

In [ ]:
ALL_WEEKS = np.arange(9, 102)
sales_lookup = normal_sales.set_index(["product_id", "store_id", "week_no"])["adj_sales"].to_dict()
promo_lookup = promo.groupby(["product_id", "store_id"])["week_no"].apply(set).to_dict()

def placebo_run(seed):
    rng = np.random.default_rng(seed)
    pair_extra, pair_weeks = [], []
    for p, s in pairs.itertuples(index=False):
        normal_weeks = np.setdiff1d(ALL_WEEKS, list(promo_lookup.get((p, s), ())))
        k = min(10, len(normal_weeks) // 4)
        fake = rng.choice(normal_weeks, size=k, replace=False)
        rest = np.setdiff1d(normal_weeks, fake)
        fake_sales = sum(sales_lookup.get((p, s, w), 0.0) for w in fake)
        baseline = sum(sales_lookup.get((p, s, w), 0.0) for w in rest) / len(rest)
        pair_extra.append(fake_sales - k * baseline)
        pair_weeks.append(k)
    pair_extra, pair_weeks = np.array(pair_extra, dtype=float), np.array(pair_weeks)
    avg = pair_extra.sum() / pair_weeks.sum()
    ci = 1.96 * pair_extra.std(ddof=1) * np.sqrt(len(pair_extra)) / pair_weeks.sum()
    return avg, ci

runs = pd.DataFrame([placebo_run(seed) for seed in range(20)], columns=["fake_extra_per_week", "ci95"])
runs["range_includes_zero"] = runs["fake_extra_per_week"].abs() <= runs["ci95"]
print(f"Average fake uplift per week: ${runs['fake_extra_per_week'].mean():.3f}")
print(f"Runs where the range includes zero: {runs['range_includes_zero'].sum()} of {len(runs)}")
print(f"Compare with the real promotions: display only ${display:.3f}, mailer only ${mailer:.3f} per week")
runs.describe().loc[["mean", "min", "max"]]

If the method is fair, the fake uplift should be close to $0 and the range should include zero in about 19 of 20 runs. That's what I'd expect by chance, with a 95% range.

## What I found

1. **The mailer drives most of the extra sales.** Mailer-only promotions created 55% of all extra sales ($161.8K of $292.0K).
2. **Display and mailer simply add up.** Display + mailer lifts a product by $0.61 per store per week, almost exactly mailer ($0.42) + display ($0.19). There's no bonus for combining them.
3. **Where the display sits doesn't matter.** All nine display locations land between $0.17 and $0.21 per week, and their ranges overlap.
4. **The front page is worth 3x the interior page.** Front-page features add $0.77 per week vs $0.26 for interior-page features, yet interior pages take half of all mailer-only weeks.
5. **Displays are the cheapest extra sales.** Display-only promotions return $1.49 of extra sales per $1 of discount; mailer-only promotions return $0.84, giving away $192.9K in discounts for $161.8K of extra sales.
6. **Mailers take some sales from similar products; displays grow the whole shelf.** Mailer-only promotions took 10% (±2.6%) of their extra sales from similar products, while display-only promotions lifted the whole sub-category 12% more than the promoted item.
7. **In a few departments, promotions mostly shift sales around.** In Floral (55%), Cosmetics (43%), Nutrition (40%) and packaged Seafood (37%), much of the "extra" was taken from similar items.
8. **No sign of stockpiling.** Sales the week after a promotion ends are slightly above normal, not below. The catch: retailers tend to promote products when demand is already rising, so some of the uplift may be timing. That's why the top recommendation needs an A/B test.
9. **The method doesn't invent uplift.** On fake promotions, the placebo test gives -$0.0002 per week, and the range includes zero in 19 of 20 runs. The real display uplift ($0.19) is about 37x bigger than that whole range.